<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/robber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```
Find Robber:
Robber is person who have done all the transactions with 10 sec interval

Points to note:
  * count over window gives count for all values enountered till now
    count will be 1, 2, 3... for each row. to get the actual count select max of it

  * select("customer_id as Robber") --does not work like this instead use
    select(col("customer_id").alias("Robber"))

  *  multipe filter conditions should be (condition) & (condition) | (condition)

  *  for time difference is seconds :  unix_timestamp(col(""))- unix_timestamp(col(""))

  * for date difference in days: datediff("transaction_time", "prev_transaction_time")
```


In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
spark

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, TimestampType, StringType, StructField
data = [
    [101, "2025-02-28 01:37:59.851", 300],
    [101, "2025-02-28 01:37:49.851", 450],
    [101, "2025-02-28 01:37:39.851", 380],
    [102, "2025-02-28 01:37:59.851", 700],
    [102, "2025-02-28 01:37:39.851", 900],
    [102, "2025-02-28 01:37:29.851", 1000],
    [103, "2025-02-28 01:37:59.851", 200],
    [102, "2025-02-28 01:37:59.851", 250]
]
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("transaction_time", StringType()),
    StructField("amount", IntegerType())]
)

transactions_data = spark.createDataFrame(data, schema)
transactions_data  = transactions_data.withColumn("transaction_time", to_timestamp("transaction_time", "yyyy-MM-dd HH:mm:ss.SSS"))
transactions_data.show(truncate=False)

+-----------+-----------------------+------+
|customer_id|transaction_time       |amount|
+-----------+-----------------------+------+
|101        |2025-02-28 01:37:59.851|300   |
|101        |2025-02-28 01:37:49.851|450   |
|101        |2025-02-28 01:37:39.851|380   |
|102        |2025-02-28 01:37:59.851|700   |
|102        |2025-02-28 01:37:39.851|900   |
|102        |2025-02-28 01:37:29.851|1000  |
|103        |2025-02-28 01:37:59.851|200   |
|102        |2025-02-28 01:37:59.851|250   |
+-----------+-----------------------+------+



In [4]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("customer_id").orderBy("transaction_time")
transactions_data_with_lag = transactions_data.withColumn("prev_transaction_time", lag("transaction_time", 1).over(window_spec))
transactions_data_with_lag.show(truncate=False)

+-----------+-----------------------+------+-----------------------+
|customer_id|transaction_time       |amount|prev_transaction_time  |
+-----------+-----------------------+------+-----------------------+
|101        |2025-02-28 01:37:39.851|380   |NULL                   |
|101        |2025-02-28 01:37:49.851|450   |2025-02-28 01:37:39.851|
|101        |2025-02-28 01:37:59.851|300   |2025-02-28 01:37:49.851|
|102        |2025-02-28 01:37:29.851|1000  |NULL                   |
|102        |2025-02-28 01:37:39.851|900   |2025-02-28 01:37:29.851|
|102        |2025-02-28 01:37:59.851|700   |2025-02-28 01:37:39.851|
|102        |2025-02-28 01:37:59.851|250   |2025-02-28 01:37:59.851|
|103        |2025-02-28 01:37:59.851|200   |NULL                   |
+-----------+-----------------------+------+-----------------------+



In [5]:
transactions_data_lag_count = transactions_data_with_lag.withColumn("transaction_duration", unix_timestamp(col("transaction_time"))- unix_timestamp(col("prev_transaction_time")))
transactions_data_lag_count.show(truncate = False)

+-----------+-----------------------+------+-----------------------+--------------------+
|customer_id|transaction_time       |amount|prev_transaction_time  |transaction_duration|
+-----------+-----------------------+------+-----------------------+--------------------+
|101        |2025-02-28 01:37:39.851|380   |NULL                   |NULL                |
|101        |2025-02-28 01:37:49.851|450   |2025-02-28 01:37:39.851|10                  |
|101        |2025-02-28 01:37:59.851|300   |2025-02-28 01:37:49.851|10                  |
|102        |2025-02-28 01:37:29.851|1000  |NULL                   |NULL                |
|102        |2025-02-28 01:37:39.851|900   |2025-02-28 01:37:29.851|10                  |
|102        |2025-02-28 01:37:59.851|700   |2025-02-28 01:37:39.851|20                  |
|102        |2025-02-28 01:37:59.851|250   |2025-02-28 01:37:59.851|0                   |
|103        |2025-02-28 01:37:59.851|200   |NULL                   |NULL                |
+---------

In [6]:
transactions_data_lag_count = transactions_data_lag_count.withColumn("window_count", count("transaction_duration").over(window_spec))
transactions_data_lag_count.show()

+-----------+--------------------+------+---------------------+--------------------+------------+
|customer_id|    transaction_time|amount|prev_transaction_time|transaction_duration|window_count|
+-----------+--------------------+------+---------------------+--------------------+------------+
|        101|2025-02-28 01:37:...|   380|                 NULL|                NULL|           0|
|        101|2025-02-28 01:37:...|   450| 2025-02-28 01:37:...|                  10|           1|
|        101|2025-02-28 01:37:...|   300| 2025-02-28 01:37:...|                  10|           2|
|        102|2025-02-28 01:37:...|  1000|                 NULL|                NULL|           0|
|        102|2025-02-28 01:37:...|   900| 2025-02-28 01:37:...|                  10|           1|
|        102|2025-02-28 01:37:...|   700| 2025-02-28 01:37:...|                  20|           3|
|        102|2025-02-28 01:37:...|   250| 2025-02-28 01:37:...|                   0|           3|
|        103|2025-02

In [7]:
transactions_data_lag_count = transactions_data_lag_count.groupBy("customer_id").agg(max("window_count").alias("total_count"),
                                                       sum(when(col("transaction_duration") == 10, 1).otherwise(0)).alias("fraudulant_transactions"))
transactions_data_lag_count.show(truncate=False)

+-----------+-----------+-----------------------+
|customer_id|total_count|fraudulant_transactions|
+-----------+-----------+-----------------------+
|101        |2          |2                      |
|102        |3          |1                      |
|103        |0          |0                      |
+-----------+-----------+-----------------------+



In [8]:
transactions_data_lag_count.filter((col("total_count") != 0) & (col("total_count") == col("fraudulant_transactions"))).select(col("customer_id").alias("Robber")).show()

+------+
|Robber|
+------+
|   101|
+------+

